In [1]:
# 1
import pandas as pd
import numpy as np
from faker import Faker
import random
from datetime import datetime, timedelta

In [2]:
# 2
fake = Faker()
num_customers = 150
transactions_per_customer = 150
customers = []
for _ in range(num_customers):
    first_name = fake.first_name()
    last_name = fake.last_name()
    customer_id = fake.unique.uuid4()  # Unique customer ID
    r = random.random()
    if r < 0.33:
        customer_type = "frequent"
    elif r < 0.66:
        customer_type = "regular"
    else:
        customer_type = "rare"
    customers.append([first_name, last_name, customer_id, customer_type])

In [3]:
# 3
df_customers = pd.DataFrame(
    customers,
    columns=['first_name', 'last_name', 'customer_id', 'customer_type']
)
transactions = []
vendors = ['Walmart', 'Amazon', 'Costco', 'Starbucks', 'American Airlines','United Airlines','CVS Pharmacy','Delta Airlines','Subway','Chick-Fil-A',
           'Chipotle','Panera Bread','Macdonalds']

# Vendor list (for retail, food chains, airlines, and pharmacy)
retail_and_airlines_vendors = ['Walmart', 'Amazon', 'Costco', 'American Airlines', 'Delta Airlines', 'United Airlines']
food_chains = ['Starbucks', 'Chipotle', 'McDonald\'s', 'Panera Bread', 'Subway', 'Chick-Fil-A']
pharmacy = ['CVS Pharmacy']

vendor_groups = {
    0: ['American Airlines', 'Walmart'],  # First 30 customers
    1: ['Delta Airlines', 'Costco'],  # Next 30 customers
    2: ['United Airlines', 'Amazon'],  # Next 30 customers
    3: ['American Airlines', 'Costco'],  # Next 30 customers
    4: ['Walmart', 'Delta Airlines'],  # Last 30 customers
}

In [4]:
# 4
print(df_customers.head())
print(df_customers.columns)
print(df_customers['customer_type'].value_counts())

  first_name last_name                           customer_id customer_type
0  Christina      Rowe  8a708e66-023b-41fa-9093-042ca5b8faea          rare
1    Jeffrey    Ibarra  613de079-c7bf-45c2-9043-35d63fa6aeb5      frequent
2       Tony     Smith  6b23af0a-9294-4f50-bab9-450a32198695          rare
3     Darius    Miller  822d2671-4972-4636-b941-7d63d4fb2674      frequent
4     Pamela     Grant  a6bab7ca-e2d3-492e-96a2-a8305f885c1e      frequent
Index(['first_name', 'last_name', 'customer_id', 'customer_type'], dtype='str')
customer_type
regular     54
rare        50
frequent    46
Name: count, dtype: int64


In [5]:
# 5

# Function to generate transaction amounts based on vendor
def generate_transaction_amount(vendor):
    if vendor in ['American Airlines', 'United Airlines', 'Delta Airlines']:
        return round(random.uniform(100.0, 500.0), 2)  # Airline vendors between $100 and $500
    elif vendor in ['Walmart', 'Costco', 'Amazon']:
        return round(random.uniform(50.0, 500.0), 2)  # Retail vendors between $50 and $500
    elif vendor in ['Starbucks', 'Chipotle', 'McDonald\'s', 'Panera Bread', 'Subway', 'Chick-Fil-A']:
        return round(random.uniform(5.0, 100.0), 2)  # Food chains between $5 and $100
    elif vendor == 'CVS Pharmacy':
        return round(random.uniform(10.0, 100.0), 2)  # CVS Pharmacy between $10 and $100
    else:
        return round(random.uniform(80.0, 500.0), 2)  # Other vendors between $80 and $500


for idx, customer in df_customers.iterrows():
    first_name = customer['first_name']
    last_name = customer['last_name']
    customer_id = customer['customer_id']
    customer_type = customer['customer_type']

    # Random starting date for the first transaction
    start_date = datetime(2024, 1, 1)
    end_date = datetime(2025, 12, 31)

    # Determine primary and secondary vendors for the customer based on their index
    primary_vendors = vendor_groups[idx // 30]  # Determine primary vendors based on index
    secondary_vendors = [vendor for vendor in vendors if vendor not in primary_vendors]

    # Create a transaction list for each customer
    all_vendors = primary_vendors + secondary_vendors

    for i in range(transactions_per_customer):
        # Select a vendor based on primary and secondary
        vendor = random.choice(all_vendors)

        # Base days between transactions by customer_type: frequent 3-7, regular 8-15, rare 16-30
        if customer_type == 'frequent':
            base_days = random.randint(3, 7)
        elif customer_type == 'regular':
            base_days = random.randint(8, 15)
        else:
            base_days = random.randint(16, 30)

        # Vendor adjustment
        if vendor in ['American Airlines', 'United Airlines', 'Delta Airlines']:
            base_days += random.randint(5, 10)
        elif vendor in ['Walmart', 'Amazon', 'Costco']:
            base_days += random.randint(-3, 3)
        elif vendor in ['Starbucks', 'Chipotle', 'McDonald\'s', 'Panera Bread', 'Subway', 'Chick-Fil-A', 'Macdonalds']:
            base_days += random.randint(-2, 2)
        elif vendor == 'CVS Pharmacy':
            base_days += random.randint(-5, 5)

        # Add noise
        base_days += random.randint(-2, 2)
        days_diff = max(0, base_days)

        transaction_date = start_date + timedelta(days=i * days_diff)
        if transaction_date > end_date:
            break

        # Generate a random transaction amount based on the vendor
        amount = generate_transaction_amount(vendor)

        # Randomly assign "Debit" or "Credit" transaction type
        transaction_type = random.choice(['Debit', 'Credit'])

        # Add transaction to the list
        transactions.append([first_name, last_name, customer_id, fake.uuid4(), transaction_date, amount, vendor, transaction_type])

# Create DataFrame from the transactions list
df_transactions = pd.DataFrame(transactions, columns=['first_name', 'last_name', 'customer_id', 'transaction_id', 'transaction_datetime', 
                                                      'amount', 'vendor', 'transaction_type'])

# View the first few transactions to verify vendor distribution
print(df_transactions[['customer_id', 'vendor', 'transaction_datetime']].head(20))





                             customer_id             vendor  \
0   8a708e66-023b-41fa-9093-042ca5b8faea  American Airlines   
1   8a708e66-023b-41fa-9093-042ca5b8faea             Costco   
2   8a708e66-023b-41fa-9093-042ca5b8faea          Starbucks   
3   8a708e66-023b-41fa-9093-042ca5b8faea        Chick-Fil-A   
4   8a708e66-023b-41fa-9093-042ca5b8faea       CVS Pharmacy   
5   8a708e66-023b-41fa-9093-042ca5b8faea             Amazon   
6   8a708e66-023b-41fa-9093-042ca5b8faea           Chipotle   
7   8a708e66-023b-41fa-9093-042ca5b8faea    United Airlines   
8   8a708e66-023b-41fa-9093-042ca5b8faea            Walmart   
9   8a708e66-023b-41fa-9093-042ca5b8faea          Starbucks   
10  8a708e66-023b-41fa-9093-042ca5b8faea     Delta Airlines   
11  8a708e66-023b-41fa-9093-042ca5b8faea       Panera Bread   
12  8a708e66-023b-41fa-9093-042ca5b8faea        Chick-Fil-A   
13  8a708e66-023b-41fa-9093-042ca5b8faea             Subway   
14  8a708e66-023b-41fa-9093-042ca5b8faea          Starb

In [ ]:
# 7
# Recompute next transaction date
df_transactions = df_transactions.sort_values(
    ['customer_id', 'transaction_datetime']
)

df_transactions['next_transaction_date'] = (
    df_transactions.groupby(['customer_id'])['transaction_datetime']
    .shift(-1)
)

df_transactions['likelihood_prediction'] = (
    df_transactions['next_transaction_date'] - df_transactions['transaction_datetime']
).dt.days

# Drop last transactions (no next date)
df_transactions = df_transactions.dropna(subset=['next_transaction_date'])

# Basic checks
print("Shape:", df_transactions.shape)
print("Date range:", df_transactions['transaction_datetime'].min(), "to", df_transactions['transaction_datetime'].max())
print("Min days:", df_transactions['likelihood_prediction'].min())
print("Max days:", df_transactions['likelihood_prediction'].max())
print("Mean days:", df_transactions['likelihood_prediction'].mean())

Shape: (5933, 10)
Date range: 2024-01-01 00:00:00 to 2025-12-27 00:00:00
Min days: 0.0
Max days: 306.0
Mean days: 16.41109051070285


In [ ]:
# 8
# Number of transactions per customer
df_transactions['customer_transaction_count'] = (
    df_transactions.groupby('customer_id')['transaction_id']
    .transform('count')
)

print(df_transactions[['customer_id', 'customer_transaction_count']].head(10))

                               customer_id  customer_transaction_count
3296  025a7907-aa17-4b51-ad39-b265e70e4951                          38
3298  025a7907-aa17-4b51-ad39-b265e70e4951                          38
3297  025a7907-aa17-4b51-ad39-b265e70e4951                          38
3300  025a7907-aa17-4b51-ad39-b265e70e4951                          38
3301  025a7907-aa17-4b51-ad39-b265e70e4951                          38
3306  025a7907-aa17-4b51-ad39-b265e70e4951                          38
3302  025a7907-aa17-4b51-ad39-b265e70e4951                          38
3299  025a7907-aa17-4b51-ad39-b265e70e4951                          38
3303  025a7907-aa17-4b51-ad39-b265e70e4951                          38
3313  025a7907-aa17-4b51-ad39-b265e70e4951                          38


In [ ]:
# 9
# How many times a customer purchased from a vendor
df_transactions['customer_vendor_txn_count'] = (
    df_transactions.groupby(['customer_id', 'vendor'])['transaction_id']
    .transform('count')
)

print(df_transactions[['customer_id', 'vendor', 'customer_vendor_txn_count']].head(10))

                               customer_id             vendor  \
3296  025a7907-aa17-4b51-ad39-b265e70e4951  American Airlines   
3298  025a7907-aa17-4b51-ad39-b265e70e4951        Chick-Fil-A   
3297  025a7907-aa17-4b51-ad39-b265e70e4951    United Airlines   
3300  025a7907-aa17-4b51-ad39-b265e70e4951        Chick-Fil-A   
3301  025a7907-aa17-4b51-ad39-b265e70e4951        Chick-Fil-A   
3306  025a7907-aa17-4b51-ad39-b265e70e4951       CVS Pharmacy   
3302  025a7907-aa17-4b51-ad39-b265e70e4951             Amazon   
3299  025a7907-aa17-4b51-ad39-b265e70e4951    United Airlines   
3303  025a7907-aa17-4b51-ad39-b265e70e4951           Chipotle   
3313  025a7907-aa17-4b51-ad39-b265e70e4951             Costco   

      customer_vendor_txn_count  
3296                          4  
3298                          3  
3297                          5  
3300                          3  
3301                          3  
3306                          2  
3302                          4  
3299        

In [9]:
# 10
# Days since customer's first transaction
df_transactions['days_since_first_purchase'] = (
    df_transactions['transaction_datetime'] -
    df_transactions.groupby('customer_id')['transaction_datetime'].transform('min')
).dt.days

print(df_transactions[['customer_id', 'days_since_first_purchase']].head(10))

                               customer_id  days_since_first_purchase
3296  025a7907-aa17-4b51-ad39-b265e70e4951                          0
3298  025a7907-aa17-4b51-ad39-b265e70e4951                         12
3297  025a7907-aa17-4b51-ad39-b265e70e4951                         22
3300  025a7907-aa17-4b51-ad39-b265e70e4951                         48
3301  025a7907-aa17-4b51-ad39-b265e70e4951                         50
3306  025a7907-aa17-4b51-ad39-b265e70e4951                         50
3302  025a7907-aa17-4b51-ad39-b265e70e4951                         54
3299  025a7907-aa17-4b51-ad39-b265e70e4951                         60
3303  025a7907-aa17-4b51-ad39-b265e70e4951                         91
3313  025a7907-aa17-4b51-ad39-b265e70e4951                        119


In [10]:
# 11
# Month and weekday features
df_transactions['transaction_month'] = df_transactions['transaction_datetime'].dt.month
df_transactions['transaction_weekday'] = df_transactions['transaction_datetime'].dt.weekday

print(df_transactions[['transaction_datetime', 'transaction_month', 'transaction_weekday']].head(10))

     transaction_datetime  transaction_month  transaction_weekday
3296           2024-01-01                  1                    0
3298           2024-01-13                  1                    5
3297           2024-01-23                  1                    1
3300           2024-02-18                  2                    6
3301           2024-02-20                  2                    1
3306           2024-02-20                  2                    1
3302           2024-02-24                  2                    5
3299           2024-03-01                  3                    4
3303           2024-04-01                  4                    0
3313           2024-04-29                  4                    0


In [ ]:
# 12
# Average spending per customer
df_transactions['customer_avg_spend'] = (
    df_transactions.groupby('customer_id')['amount']
    .transform('mean')
)

print(df_transactions[['customer_id', 'amount', 'customer_avg_spend']].head(10))

                               customer_id  amount  customer_avg_spend
3296  025a7907-aa17-4b51-ad39-b265e70e4951  226.86          204.265263
3298  025a7907-aa17-4b51-ad39-b265e70e4951   61.46          204.265263
3297  025a7907-aa17-4b51-ad39-b265e70e4951  196.94          204.265263
3300  025a7907-aa17-4b51-ad39-b265e70e4951   92.76          204.265263
3301  025a7907-aa17-4b51-ad39-b265e70e4951   52.69          204.265263
3306  025a7907-aa17-4b51-ad39-b265e70e4951   69.03          204.265263
3302  025a7907-aa17-4b51-ad39-b265e70e4951   83.97          204.265263
3299  025a7907-aa17-4b51-ad39-b265e70e4951  272.10          204.265263
3303  025a7907-aa17-4b51-ad39-b265e70e4951   13.84          204.265263
3313  025a7907-aa17-4b51-ad39-b265e70e4951  233.16          204.265263


In [12]:
# 13
# Average time gap between purchases for each customer
df_transactions['customer_avg_gap'] = (
    df_transactions.groupby('customer_id')['likelihood_prediction']
    .transform('mean')
)

print(df_transactions[['customer_id','likelihood_prediction','customer_avg_gap']].head(10))

                               customer_id  likelihood_prediction  \
3296  025a7907-aa17-4b51-ad39-b265e70e4951                   12.0   
3298  025a7907-aa17-4b51-ad39-b265e70e4951                   10.0   
3297  025a7907-aa17-4b51-ad39-b265e70e4951                   26.0   
3300  025a7907-aa17-4b51-ad39-b265e70e4951                    2.0   
3301  025a7907-aa17-4b51-ad39-b265e70e4951                    0.0   
3306  025a7907-aa17-4b51-ad39-b265e70e4951                    4.0   
3302  025a7907-aa17-4b51-ad39-b265e70e4951                    6.0   
3299  025a7907-aa17-4b51-ad39-b265e70e4951                   31.0   
3303  025a7907-aa17-4b51-ad39-b265e70e4951                   28.0   
3313  025a7907-aa17-4b51-ad39-b265e70e4951                   13.0   

      customer_avg_gap  
3296              17.0  
3298              17.0  
3297              17.0  
3300              17.0  
3301              17.0  
3306              17.0  
3302              17.0  
3299              17.0  
3303           

In [13]:
# 14
# Remove rows where likelihood_prediction is NaN
df_transactions = df_transactions.dropna(subset=['likelihood_prediction'])

print("Dataset shape after cleaning:", df_transactions.shape)

Dataset shape after cleaning: (5933, 17)


In [14]:
# 15
# Features for model
# vendor_* columns are added in PIPELINE 2/4 (below). If you run this cell before that,
# we use the base feature list only; re-run after PIPELINE 2/4 for the full set.
FEATURES_FULL = [
    'amount',
    'customer_transaction_count',
    'customer_vendor_txn_count',
    'days_since_first_purchase',
    'transaction_month',
    'transaction_weekday',
    'customer_avg_spend',
    'customer_avg_gap',
    'vendor_preferred_day',
    'vendor_day_std',
    'vendor_preferred_month',
    'vendor_month_std',
    'vendor_monthly_frequency',
]
FEATURES_BASE = [
    'amount',
    'customer_transaction_count',
    'customer_vendor_txn_count',
    'days_since_first_purchase',
    'transaction_month',
    'transaction_weekday',
    'customer_avg_spend',
    'customer_avg_gap',
]

if 'vendor_preferred_day' in df_transactions.columns:
    features = FEATURES_FULL
else:
    features = FEATURES_BASE
    print(
        "Note: vendor pattern columns are not in df_transactions yet. "
        "Run PIPELINE 1/4 and 2/4 below, then re-run this cell to include them."
    )

X = df_transactions[features]
y = df_transactions['likelihood_prediction']

print(X.head())
print("Target sample:")
print(y.head())

Note: vendor pattern columns are not in df_transactions yet. Run PIPELINE 1/4 and 2/4 below, then re-run this cell to include them.
      amount  customer_transaction_count  customer_vendor_txn_count  \
3296  226.86                          38                          4   
3298   61.46                          38                          3   
3297  196.94                          38                          5   
3300   92.76                          38                          3   
3301   52.69                          38                          3   

      days_since_first_purchase  transaction_month  transaction_weekday  \
3296                          0                  1                    0   
3298                         12                  1                    5   
3297                         22                  1                    1   
3300                         48                  2                    6   
3301                         50                  2                

In [ ]:
# 16
print("Shape:", df_transactions.shape)
print("Date range:", 
      df_transactions['transaction_datetime'].min(), 
      "to", 
      df_transactions['transaction_datetime'].max())

print(df_transactions['likelihood_prediction'].describe())

Shape: (5933, 17)
Date range: 2024-01-01 00:00:00 to 2025-12-27 00:00:00
count    5933.000000
mean       16.411091
std        21.562773
min         0.000000
25%         4.000000
50%        10.000000
75%        21.000000
max       306.000000
Name: likelihood_prediction, dtype: float64


In [16]:
# 17
print("Date range:", df_transactions['transaction_datetime'].min(), "to", df_transactions['transaction_datetime'].max())
print("Min likelihood:", df_transactions['likelihood_prediction'].min())
print("Max likelihood:", df_transactions['likelihood_prediction'].max())
print("Any negative?", (df_transactions['likelihood_prediction'] < 0).any())

Date range: 2024-01-01 00:00:00 to 2025-12-27 00:00:00
Min likelihood: 0.0
Max likelihood: 306.0
Any negative? False


In [17]:
# 18
print(df_transactions['likelihood_prediction'].describe())
print(df_transactions.groupby('customer_id')['likelihood_prediction'].mean().head())
print(df_transactions.groupby('vendor')['likelihood_prediction'].mean())

count    5933.000000
mean       16.411091
std        21.562773
min         0.000000
25%         4.000000
50%        10.000000
75%        21.000000
max       306.000000
Name: likelihood_prediction, dtype: float64
customer_id
025a7907-aa17-4b51-ad39-b265e70e4951    17.000000
02a391c5-8198-463f-a5d2-764b97a172e9    19.555556
0314d1c1-488a-43d8-aeba-1430a9d6aa32    12.000000
0384a7ba-1121-44cc-b0ae-fc8321d3beeb    21.333333
03af64b8-c29b-41c9-a7b6-a113e108d422    14.520000
Name: likelihood_prediction, dtype: float64
vendor
Amazon               14.664384
American Airlines    20.863184
CVS Pharmacy         14.828512
Chick-Fil-A          15.875764
Chipotle             15.868085
Costco               15.313175
Delta Airlines       20.224806
Macdonalds           15.382353
Panera Bread         15.365854
Starbucks            16.074153
Subway               14.759013
United Airlines      20.378251
Walmart              15.519669
Name: likelihood_prediction, dtype: float64


In [18]:
# 19
same_day_transactions = df_transactions[df_transactions.duplicated(
    subset=['customer_id', 'vendor', 'transaction_datetime'], keep=False
)]
print(same_day_transactions[['customer_id', 'vendor', 'transaction_datetime']].head(10))

                               customer_id        vendor transaction_datetime
5829  0314d1c1-488a-43d8-aeba-1430a9d6aa32       Walmart           2024-01-01
5874  0314d1c1-488a-43d8-aeba-1430a9d6aa32       Walmart           2024-01-01
1196  045160a6-9748-4cfa-8d1a-9ebe7693189c       Walmart           2024-01-17
1198  045160a6-9748-4cfa-8d1a-9ebe7693189c       Walmart           2024-01-17
288   05932030-10a4-4744-8657-212f157456e9  CVS Pharmacy           2024-01-01
323   05932030-10a4-4744-8657-212f157456e9  CVS Pharmacy           2024-01-01
2522  0747c0c8-e442-406d-824e-1a05db18947a  CVS Pharmacy           2024-01-01
2570  0747c0c8-e442-406d-824e-1a05db18947a  CVS Pharmacy           2024-01-01
4465  08789119-de9b-4d61-a81a-44613a33d5d8       Walmart           2024-01-01
4486  08789119-de9b-4d61-a81a-44613a33d5d8       Walmart           2024-01-01


## Export pipeline (4 code cells below)

Order: **data generation** (above) → **PIPELINE 1/4** (sort + vendor-level likelihood) → **PIPELINE 2/4** (feature engineering) → **PIPELINE 3/4** (in-memory sanity) → **PIPELINE 4/4** (**export CSV — run last**).


In [19]:
# PIPELINE 1/4 — Vendor-level likelihood (same customer + same vendor)
# Recompute likelihood at customer+vendor level (days until next purchase at same vendor)
df_transactions = df_transactions.sort_values(
    ['customer_id', 'vendor', 'transaction_datetime']
).reset_index(drop=True)

df_transactions['next_transaction_date'] = df_transactions.groupby(
    ['customer_id', 'vendor']
)['transaction_datetime'].shift(-1)

df_transactions['likelihood_prediction'] = (
    df_transactions['next_transaction_date'] - df_transactions['transaction_datetime']
).dt.days

# Drop rows with no next transaction (last purchase per customer+vendor)
df_transactions = df_transactions.dropna(subset=['next_transaction_date'])

# Recompute customer_avg_gap on clean data
df_transactions['customer_avg_gap'] = (
    df_transactions.groupby('customer_id')['likelihood_prediction']
    .transform('mean')
)

print("Shape:", df_transactions.shape)
print("Min likelihood:", df_transactions['likelihood_prediction'].min())
print("Max likelihood:", df_transactions['likelihood_prediction'].max())
print("Any negative?", (df_transactions['likelihood_prediction'] < 0).any())
print(df_transactions['likelihood_prediction'].describe())


Shape: (4126, 17)
Min likelihood: 0.0
Max likelihood: 667.0
Any negative? False
count    4126.000000
mean      101.523267
std        99.613894
min         0.000000
25%        28.000000
50%        70.000000
75%       143.000000
max       667.000000
Name: likelihood_prediction, dtype: float64


In [ ]:
# PIPELINE 2/4 — Temporal + vendor features (expanded)
# Temporal features — what just happened (not all-time history)

df_transactions = df_transactions.sort_values(['customer_id', 'transaction_datetime']).reset_index(drop=True)

df_transactions['days_since_last_transaction'] = (
    df_transactions['transaction_datetime'] -
    df_transactions.groupby('customer_id')['transaction_datetime'].shift(1)
).dt.days

df_transactions['lag_1_gap'] = df_transactions.groupby('customer_id')['likelihood_prediction'].shift(1)
df_transactions['lag_2_gap'] = df_transactions.groupby('customer_id')['likelihood_prediction'].shift(2)
df_transactions['lag_3_gap'] = df_transactions.groupby('customer_id')['likelihood_prediction'].shift(3)

df_transactions['rolling_avg_gap_7'] = (
    df_transactions.groupby('customer_id')['likelihood_prediction']
    .transform(lambda x: x.shift(1).rolling(7, min_periods=2).mean())
)
df_transactions['rolling_avg_gap_3'] = (
    df_transactions.groupby('customer_id')['likelihood_prediction']
    .transform(lambda x: x.shift(1).rolling(3, min_periods=2).mean())
)
df_transactions['rolling_avg_gap_14'] = (
    df_transactions.groupby('customer_id')['likelihood_prediction']
    .transform(lambda x: x.shift(1).rolling(14, min_periods=5).mean())
)

df_transactions['gap_trend_short'] = df_transactions['rolling_avg_gap_3'] - df_transactions['rolling_avg_gap_7']
df_transactions['gap_trend_long'] = df_transactions['rolling_avg_gap_7'] - df_transactions['rolling_avg_gap_14']

df_transactions['rolling_spend_3'] = (
    df_transactions.groupby('customer_id')['amount']
    .transform(lambda x: x.shift(1).rolling(3, min_periods=2).mean())
)
df_transactions['rolling_spend_7'] = (
    df_transactions.groupby('customer_id')['amount']
    .transform(lambda x: x.shift(1).rolling(7, min_periods=3).mean())
)
df_transactions['spend_trend'] = df_transactions['rolling_spend_3'] - df_transactions['rolling_spend_7']

vendor_avg_gap = df_transactions.groupby(['customer_id', 'vendor'])['likelihood_prediction'].transform('mean')
df_transactions['vendor_gap_vs_avg'] = vendor_avg_gap - df_transactions['customer_avg_gap']

vendor_avg_spend = df_transactions.groupby(['customer_id', 'vendor'])['amount'].transform('mean')
df_transactions['vendor_spend_vs_avg'] = vendor_avg_spend - df_transactions['customer_avg_spend']

vendor_category_map = {
    'American Airlines': 'airline', 'Delta Airlines': 'airline', 'United Airlines': 'airline',
    'Walmart': 'retail', 'Amazon': 'retail', 'Costco': 'retail',
    'Starbucks': 'food', 'Chipotle': 'food', 'Subway': 'food',
    'Chick-Fil-A': 'food', 'Panera Bread': 'food', 'Macdonalds': 'food',
    'CVS Pharmacy': 'pharmacy'
}
cat_map = {'food': 0, 'pharmacy': 1, 'retail': 2, 'airline': 3, 'other': 4}
df_transactions['vendor_category'] = df_transactions['vendor'].map(vendor_category_map).fillna('other')
df_transactions['vendor_category_code'] = df_transactions['vendor_category'].map(cat_map)

# --- Purchase pattern features (day of month + seasonality) ---

# Average day-of-month this customer buys from this vendor
df_transactions['transaction_day'] = pd.to_datetime(
    df_transactions['transaction_datetime']
).dt.day

df_transactions['vendor_preferred_day'] = (
    df_transactions.groupby(['customer_id', 'vendor'])['transaction_day']
    .transform('mean')
).round(0)

# How consistent is the day-of-month (low std = very regular buyer)
df_transactions['vendor_day_std'] = (
    df_transactions.groupby(['customer_id', 'vendor'])['transaction_day']
    .transform('std')
).fillna(0)

# Which months does this customer buy from this vendor
df_transactions['vendor_preferred_month'] = (
    df_transactions.groupby(['customer_id', 'vendor'])['transaction_month']
    .transform('mean')
).round(0)

# Month consistency
df_transactions['vendor_month_std'] = (
    df_transactions.groupby(['customer_id', 'vendor'])['transaction_month']
    .transform('std')
).fillna(0)

# Purchase frequency per month (how many times per month on average)
df_transactions['vendor_monthly_frequency'] = (
    df_transactions.groupby(['customer_id', 'vendor'])['transaction_id']
    .transform('count') /
    df_transactions.groupby(['customer_id', 'vendor'])['transaction_month']
    .transform('nunique')
).round(2)

print("Pattern features added:")
print(df_transactions[[
    'vendor', 'vendor_preferred_day', 'vendor_day_std',
    'vendor_preferred_month', 'vendor_month_std', 'vendor_monthly_frequency'
]].head(10))

# dropna — require enough history for lags and rolling windows
df_transactions = df_transactions.dropna(subset=[
    'days_since_last_transaction', 'lag_1_gap', 'lag_2_gap', 'lag_3_gap',
    'rolling_avg_gap_3', 'rolling_avg_gap_7', 'rolling_avg_gap_14',
    'rolling_spend_3', 'rolling_spend_7',
])

print("Shape after temporal features + dropna:", df_transactions.shape)
print(df_transactions[[
    'days_since_last_transaction', 'lag_1_gap', 'lag_2_gap', 'lag_3_gap',
    'rolling_avg_gap_3', 'rolling_avg_gap_7', 'rolling_avg_gap_14',
    'gap_trend_short', 'gap_trend_long', 'rolling_spend_3', 'rolling_spend_7', 'spend_trend',
    'vendor_gap_vs_avg', 'vendor_spend_vs_avg', 'vendor_category_code',
]].describe())


Pattern features added:
              vendor  vendor_preferred_day  vendor_day_std  \
0             Amazon                  26.0        2.516611   
1    United Airlines                   3.0        3.214550   
2           Chipotle                   7.0        7.211103   
3             Costco                  28.0        1.414214   
4  American Airlines                  20.0       10.606602   
5           Chipotle                   7.0        7.211103   
6           Chipotle                   7.0        7.211103   
7             Subway                   8.0        3.535534   
8             Subway                   8.0        3.535534   
9            Walmart                  14.0        6.110101   

   vendor_preferred_month  vendor_month_std  vendor_monthly_frequency  
0                     8.0          5.196152                       1.5  
1                     8.0          4.725816                       1.0  
2                     5.0          1.000000                       1.0  
3    

In [ ]:
# PIPELINE 3/4 — In-memory sanity check (before export)
print("Ready to export — shape:", df_transactions.shape)
print("Min likelihood:", df_transactions['likelihood_prediction'].min())
print("Max likelihood:", df_transactions['likelihood_prediction'].max())
print("Mean likelihood:", df_transactions['likelihood_prediction'].mean())
print("Columns:", list(df_transactions.columns))


Ready to export — shape: (3376, 39)
Min likelihood: 0.0
Max likelihood: 624.0
Mean likelihood: 95.31635071090048
Columns: ['first_name', 'last_name', 'customer_id', 'transaction_id', 'transaction_datetime', 'amount', 'vendor', 'transaction_type', 'next_transaction_date', 'likelihood_prediction', 'customer_transaction_count', 'customer_vendor_txn_count', 'days_since_first_purchase', 'transaction_month', 'transaction_weekday', 'customer_avg_spend', 'customer_avg_gap', 'days_since_last_transaction', 'lag_1_gap', 'lag_2_gap', 'lag_3_gap', 'rolling_avg_gap_7', 'rolling_avg_gap_3', 'rolling_avg_gap_14', 'gap_trend_short', 'gap_trend_long', 'rolling_spend_3', 'rolling_spend_7', 'spend_trend', 'vendor_gap_vs_avg', 'vendor_spend_vs_avg', 'vendor_category', 'vendor_category_code', 'transaction_day', 'vendor_preferred_day', 'vendor_day_std', 'vendor_preferred_month', 'vendor_month_std', 'vendor_monthly_frequency']


In [ ]:
# PIPELINE 4/4 — Export CSV (last step; run after all feature engineering)
import os
expected = [
    'lag_3_gap', 'rolling_avg_gap_14', 'gap_trend_short', 'gap_trend_long',
    'spend_trend', 'rolling_spend_7', 'vendor_gap_vs_avg',
    'vendor_spend_vs_avg', 'vendor_category_code'
]
still_missing = [c for c in expected if c not in df_transactions.columns]
if still_missing:
    print("ERROR — still missing:", still_missing)
else:
    os.makedirs("predictor/data", exist_ok=True)
    df_transactions.to_csv("predictor/data/dataset1.csv", index=False)
    print("CSV exported successfully")
    print("Shape:", df_transactions.shape)
    print("Columns:", len(df_transactions.columns))


CSV exported successfully
Shape: (3376, 39)
Columns: 39
